# Quantifying alternative splicing with LeafCutter2

Quantifies alternative splicing from aligned RNA-seq with LeafCutter2, producing intron-usage tables in which every junction carries a productive or unproductive label.

## Overview

This module turns aligned RNA-seq into splicing phenotypes, as `splicing_calling` does, but replaces LeafCutter with LeafCutter2 and adds a functional label to every junction. It expects BAM files already mapped with STAR using the WASP option, following the [GTEx/TOPMed RNA-seq pipeline](https://github.com/broadinstitute/gtex-pipeline/blob/master/TOPMed_RNAseq_pipeline.md).

Splicing is quantified as alternatively excised intron usage: junctions sharing a splice site are grouped into clusters and each intron is reported as a fraction of the reads in its cluster, so skipped exons, alternative 5-prime and 3-prime splice site usage and more complex events are captured by one measure without naming the event type [Li et al. (2018)](https://doi.org/10.1038/s41588-017-0004-9).

What LeafCutter2 adds is classification. A dynamic programming pass over the clusters reads each junction against the annotated coding sequence and asks whether the transcript it implies stays in frame to a stop codon: junctions are labelled `PR` (productive, protein-coding), `UP` (unproductive, premature stop and so a nonsense-mediated decay target), `NE` (ambiguous) or `IN` (intergenic). The label is appended to the intron ID, so `chr10:134786:179993:clu_1_+:PR` carries chromosome, intron, cluster and function in one key ([Dai et al., 2025](https://doi.org/10.1101/2025.04.06.646893)). Unproductive splicing is the quantity of interest wherever a splicing factor is suspected of losing control of cryptic exons.

- `leafcutter2` extracts junctions from each BAM with regtools, then clusters, classifies and quantifies them.
- `leafcutter2_preprocessing` is the first half of `leafcutter2`, stopping once the list of per-sample `.junc` files is written. Run it only when the clustering happens elsewhere.
- `leafcutter2_cluster` starts from an existing list of junction files and does the clustering, classification and quantification alone. It is the entry point when junctions were extracted by another pipeline.
- `star2junc` converts STAR `SJ.out.tab` files into the junction BED format LeafCutter2 reads, for cohorts where the BAMs are no longer available.

`leafcutter2` and `leafcutter2_preprocessing` share their first step. `leafcutter2_cluster` is an alternative entry point to the second step, not a stage that follows it.

**When to run it.** After STAR alignment, and before `splicing_normalization`, in place of `splicing_calling`.

## Input

- `--samples` **`tests/fixtures/splicing_calling/leafcutter/protocol_example.sample_list.txt`**
(the sample manifest, tab delimited with a header, shared with `splicing_calling`. `coord_bam_list` is the column read for BAM paths; `SJ_list` is read only by `star2junc`. Strandness is `rf`, `fr`, or `strand_missing`. Output files take their prefix from this filename with all extensions stripped, so this manifest produces `protocol_example.*`; override it with `--output-prefix`.)

```
sample_id	strand	coord_bam_list	SJ_list
SAMPLE_001	rf	SAMPLE_001.Aligned.sortedByCoord.out.bam	SAMPLE_001.SJ.out.tab
SAMPLE_002	rf	SAMPLE_002.Aligned.sortedByCoord.out.bam	SAMPLE_002.SJ.out.tab
```

The `SJ.out.tab` files this manifest names are shipped, so `star2junc` runs as written; the junction files it pairs with are shipped too, so `leafcutter2_cluster` does as well. The BAMs are not in the repository — supply them through `--data-dir` to run `leafcutter2` or `leafcutter2_preprocessing` from this manifest, or use the BAM manifest below.

- `--samples` **`tests/fixtures/splicing_calling/leafcutter/protocol_example.bam_list.txt`**
(a second manifest for the workflows that start from BAMs, naming the one alignment the repository ships. Its strandness is `XS`, the mode `regtools` infers from the BAM's own tags, because this fixture carries no library strand information.)

```
sample_id	strand	coord_bam_list	SJ_list
SAMPLE_001	XS	protocol_example.chr22_16M_17M.bam	NA
```

That BAM is `tests/fixtures/phenotype_formatting/protocol_example.chr22_16M_17M.bam`, the chr22:16M-17M alignment the repository's own `splicing_calling` tests use, and the one the shipped `.junc` fixtures were extracted from. Junction files are named after the BAM rather than the `sample_id`, so it produces `protocol_example.chr22_16M_17M.junc`.

* `--data-dir` -- directory holding the files named in the manifest; defaults to the directory containing `--samples`.
* `--cwd` -- output directory, default `output`.
* `--container` -- default empty. LeafCutter2 is installed into the pixi environment rather than distributed as an image, so this module runs without a container unless one is supplied.

**LeafCutter2 2.0 or newer is required.** The 2.x package restructured the code into `leafcutter2/` with `classifier.py` and a `cli.py` entry point, and added the `leafcutter2-star2junc` command that `star2junc` calls. Pre-2.0 builds ship the original repository scripts (`Transcript_tools.py`, `ForwardSpliceJunctionClassifier.py`) with no package metadata, and their classification step fails.

Check by whether `leafcutter2-star2junc` is on `PATH`, not by the version string: packaged builds do not always carry the upstream version. The `dnachun` conda build is labelled `1.0.1` but ships 2.0.1, so a version comparison alone would wrongly reject it.

`pixi global install leafcutter2` installs it. LeafCutter2 is not published on PyPI, so `pip install leafcutter2` fails; to install from source use `pip install git+https://github.com/leafcutter2/leafcutter2.git`, or put that URL in a project manifest's `[pypi-dependencies]`.

Junction extraction (`leafcutter2`, `leafcutter2_preprocessing`):

* `--anchor-len` -- minimum anchor length on each side of a junction, default `8`.
* `--min-intron-len` -- shortest intron considered, default `50`.
* `--max-intron-len` -- longest intron considered, default `500000`.

Clustering, classification and quantification (`leafcutter2`, `leafcutter2_cluster`):

* `--annotation-gtf` **`reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.ERCC.gtf`**
(GTF carrying `gene`, `transcript`, `CDS`, `start_codon` and `stop_codon` features. Classification reads the coding sequence from it, so a GTF without start and stop codons will not do.)

* `--genome-fasta` **`reference_data/GRCh38_full_analysis_set_plus_decoy_hla.noALT_noHLA_noDecoy_ERCC.fasta`**
(genome assembly matching the GTF, with a `.fai` index beside it)

- `--juncfiles` **`tests/fixtures/splicing_calling/leafcutter/protocol_example.junction_files.txt`**
(`leafcutter2_cluster` only: one junction file path per line)

```
tests/fixtures/splicing_calling/leafcutter/SAMPLE_001.junc.gz
tests/fixtures/splicing_calling/leafcutter/SAMPLE_002.junc.gz
```

* `--min-clu-reads` -- minimum reads in a cluster, default `30`.
* `--min-reads` -- minimum reads for a junction to be considered, default `5`.
* `--min-clu-ratio` -- minimum fraction of cluster reads supporting a junction, default `0.001`.
* `--min-read-std` -- minimum standard deviation of reads across samples, default `0.5`.
* `--max-intron-len` -- longest intron clustered, default `500000`.
* `--offset` -- coordinate offset. Defaults to `1` in `leafcutter2`; **required, with no default, in `leafcutter2_cluster`**. See below.
* `--output-prefix` -- prefix for every output file, default the `--samples` name with all extensions stripped. Set it when two workflows write into one `--cwd`, or when the manifest name carries dots that are not extensions.

`--max-intron-len` is `500000` here against a LeafCutter2 default of `100000`. The larger value is the GTEx sQTL discovery default used throughout this protocol, and the one `splicing_calling` applies. Leaving it at `100000` silently discards every intron above that length, so the two pipelines would no longer be comparable. The reason for the larger value is protocol compatibility, not a claim about where cryptic exons occur.

`--offset` depends on the junction file format, and getting it wrong corrupts the classification rather than failing loudly.

Junction files come in two shapes. regtools `.junc` is BED12, with the anchor blocks in columns 11 and 12; the BED6 form written by `star2junc`, and used by LeafCutter2's own example data, carries intron coordinates directly. The two need different offsets:

| Input | Offset | Evidence |
| --- | --- | --- |
| regtools `.junc` (BED12) | `1` | Clustering the repository's two junction fixtures gives no intron shared with `leafcutter_cluster_regtools.R` at offset `0`, and exact agreement on all six at offset `1`. |
| BED6, including `star2junc` output | `0` | LeafCutter2's twelve-sample chr10 example classifies 2,324 `PR` / 345 `UP` / 236 `NE` / 125 `IN` at offset `0`, with intron IDs matching the published ones. At offset `1` the same data gives 505 / 1,261 / 1,138 / 125. Confirmed on three whole-transcriptome brain samples against Ensembl 108: 55,248 junctions, 68% `PR`. |

`leafcutter2` defaults to `1`, because its junctions always come from regtools via `leafcutter2_1`, so the format is known.

`leafcutter2_cluster` has no default and will not run without `--offset`. Its junction list comes from outside the module, so neither value is safe to assume: a default would be wrong roughly half the time and wrong silently. Pass `0` for BED6, `1` for regtools `.junc`, as the minimal working example does.

The step rejects a junction list that mixes the two formats, since one offset cannot suit both. Beyond that, an offset error is silent: the symptom is a collapse in the productive fraction, because shifted coordinates no longer meet the annotated splice sites. Check the `PR` share before trusting a run.

The GTF and FASTA are not in the repository; both are built by [`reference_data_preparation`](https://github.com/statfungen/xqtl-protocol/blob/main/code/reference_data/reference_data_preparation.ipynb) and are the same pair `RNA_calling` and `bulk_expression_normalization` use. Chromosome names must match across the junction files, the GTF and the FASTA. This pair is `chr`-prefixed, so junctions called against a bare-numbered assembly will not classify; the clustering steps check the two naming styles agree before starting and stop if they do not.

Runtime:

* `--numThreads`, `--job-size`, `--walltime`, `--mem` -- threads (default `8`) and cluster resources.
* `--modular-script-dir` -- location of the shell drivers, default `code/script`.

## Output

- **`output/leafcutter2/SAMPLE_001.junc`** (`leafcutter2_1`)
(per-sample junctions extracted from the BAM by regtools, BED12 with the anchor blocks in columns 11 and 12)

- **`output/leafcutter2/protocol_example.junction_counts.gz`** (`leafcutter2_2`, `leafcutter2_cluster`)
(one row per junction, keyed `chrom:start:end:clu_N_strand:CLASS`, giving raw supporting-read counts per sample. This is the table LeafCutter's R differential tools read.)

```
chrom SAMPLE_001 SAMPLE_002
chr22:12314919:12319407:clu_1_-:IN 31 23
chr22:12314919:12351352:clu_1_-:IN 25 0
chr22:28168701:28196318:clu_2_-:UP 11 6
```

- **`output/leafcutter2/protocol_example.cluster_ratios.gz`** (`leafcutter2_2`, `leafcutter2_cluster`)
(the same rows as `numerator/denominator` strings, the junction's reads over its cluster's reads)

```
chrom SAMPLE_001 SAMPLE_002
chr22:12314919:12319407:clu_1_-:IN 31/56 23/23
chr22:12314919:12351352:clu_1_-:IN 25/56 0/23
```

- **`output/leafcutter2/clustering/protocol_example_clusters`** (`leafcutter2_2`, `leafcutter2_cluster`)
(the intron cluster definitions, one cluster per line as `start:end:reads`. Passing this back through the tool's `--cluster` flag skips clustering on a repeat run.)

```
chr22:- 12314919:12319407:54 12314919:12351352:25
chr22:- 28168701:28196318:17 28168701:28214161:15
```

- **`output/leafcutter2/clustering/protocol_example_junction_classifications.txt`** (`leafcutter2_2`, `leafcutter2_cluster`)
(the classification table the `:CLASS` label is derived from. `Gene_name` carries whatever the GTF's `gene_name` tag holds, which in the protocol's reformatted annotation is the Ensembl gene ID.)

```
Gene_name	Intron_coord	Strand	Annot	Coding	UTR	GencodePC
ENSG00000100154	chr22:28629830-28679621	-	True	True	False	True
ENSG00000100154	chr22:28306643-28629551	-	True	True	False	True
```

- **`output/leafcutter2/clustering/protocol_example_exon_stats.txt`** (`leafcutter2_2`, `leafcutter2_cluster`)
(per-exon counts from the classification pass)

- **`output/leafcutter2/clustering/protocol_example_lowusage_introns`** (`leafcutter2_2`, `leafcutter2_cluster`)
(introns dropped for falling below `--min-clu-ratio`)

- **`output/leafcutter2/clustering/protocol_example_nuc_rule_distances.txt`** (`leafcutter2_2`, `leafcutter2_cluster`)
(distances used by the 50-nucleotide NMD rule)

- **`output/leafcutter2/clustering/protocol_example_long_exon_distances.txt`** (`leafcutter2_2`, `leafcutter2_cluster`)
(distances used by the long-final-exon exception to that rule)

Names under `clustering/` and `leafcutter1_files/` are LeafCutter2's own, built by appending an underscore and a suffix to `--outprefix`; this module cannot change them. The files it names itself use a dot.

- **`output/leafcutter2/leafcutter1_files/protocol_example_perind.counts.gz`** (`leafcutter2_2`, `leafcutter2_cluster`)
(LeafCutter1-format cluster ratios without the class label, written because this module always passes `--keepleafcutter1`. It lets the original `leafcutter_ds.R` run on LeafCutter2 clusters.)

- **`output/leafcutter2/leafcutter1_files/protocol_example_perind_numers.counts.gz`** (`leafcutter2_2`, `leafcutter2_cluster`)
(the matching LeafCutter1-format numerator table)

- **`output/leafcutter2/protocol_example.intron_usage_perind.junc`** (`leafcutter2_preprocessing_2`)
(the list of per-sample `.junc` files, one path per line)

- **`output/leafcutter2/SAMPLE_001.SJ.out.junc.gz`** (`star2junc`)
(BED6 junctions converted from a STAR `SJ.out.tab`, read counts in column 5. The name comes from the `SJ.out.tab` it was converted from.)

```
chr22 12314919 12319407 . 31 -
chr22 28168701 28196318 . 11 -
```

Sample column names in `junction_counts.gz` and `cluster_ratios.gz` are the junction files' base names, not the `sample_id` column of the manifest. Junctions converted by `star2junc` therefore carry names like `SAMPLE_001.SJ.out`, and real STAR output often adds a pass suffix, giving `SAMPLE_001.1pass.SJ.out`. A groups file for differential splicing must use these names, not the manifest's.

Each step also writes a `.stderr` log beside its output. LeafCutter2 logs to standard error, so the corresponding `.stdout` is empty and is not kept.

**`protocol_example.cluster_ratios.gz` is the file that feeds `splicing_normalization`**, not `junction_counts.gz`. Its `prepare_phenotype` step splits every cell on `/` and reads the denominator, so the `numerator/denominator` form is required; handing it the integer counts fails on the missing second field. `junction_counts.gz` is for differential splicing tools that want raw counts, such as LeafCutter's `leafcutter_ds.R`.

The fifth `:CLASS` field in the intron ID is safe for that step: it reads only the first three colon-separated fields as coordinates and carries the full ID through unchanged.

## Minimal Working Example

### Cluster and classify from junction files

`leafcutter2_cluster` takes a list of junction files and runs clustering, classification and quantification. The repository ships two regtools junction fixtures, so this is the one workflow that runs on repository data; the GTF and FASTA are not shipped and must be supplied. The fixtures cover chr22, so the reference must include that chromosome and use `chr`-prefixed names. `--offset 1` is passed because the fixtures are regtools `.junc`. Almost all of the runtime is parsing the annotation, not the junctions, so it is near-constant regardless of cohort size; peak memory was 3.3 GB against a 1.3 GB GTF. Four of the six introns in `expected_perind_numers.counts.gz` survive, at identical coordinates; the other two belong to a 12-read cluster dropped by the 30-read `--min-clu-reads` minimum.

**Timing**: ~7 min (on toy dataset)

In [ ]:
sos run pipeline/splicing_calling_leafcutter2.ipynb leafcutter2_cluster \
    --cwd output/leafcutter2 \
    --samples tests/fixtures/splicing_calling/leafcutter/protocol_example.sample_list.txt \
    --juncfiles tests/fixtures/splicing_calling/leafcutter/protocol_example.junction_files.txt \
    --annotation-gtf reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.ERCC.gtf \
    --genome-fasta reference_data/GRCh38_full_analysis_set_plus_decoy_hla.noALT_noHLA_noDecoy_ERCC.fasta \
    --offset 1

### Junction extraction and clustering from BAMs

`leafcutter2` extracts junctions from every BAM with regtools and then clusters, classifies and quantifies them. This is the entry point for a full cohort. The one alignment the repository ships covers chr22:16M-17M in a single sample, so `--min-clu-reads 3` is needed here: at the default of 30 no cluster survives, and LeafCutter2 then fails in classification rather than exiting cleanly. The repository's own `splicing_calling` test uses the same lowered value on this data. A real cohort should leave the default alone.

**Timing**: ~7 min (on toy dataset)

In [ ]:
sos run pipeline/splicing_calling_leafcutter2.ipynb leafcutter2 \
    --cwd output/leafcutter2 \
    --samples tests/fixtures/splicing_calling/leafcutter/protocol_example.bam_list.txt \
    --data-dir tests/fixtures/phenotype_formatting \
    --annotation-gtf reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.ERCC.gtf \
    --genome-fasta reference_data/GRCh38_full_analysis_set_plus_decoy_hla.noALT_noHLA_noDecoy_ERCC.fasta \
    --min-clu-reads 3

### Junction extraction only

`leafcutter2_preprocessing` is the first half of the command above, stopping once the list of `.junc` files is written.

**Timing**: ~1 min (on toy dataset)

In [ ]:
sos run pipeline/splicing_calling_leafcutter2.ipynb leafcutter2_preprocessing \
    --cwd output/leafcutter2 \
    --samples tests/fixtures/splicing_calling/leafcutter/protocol_example.bam_list.txt \
    --data-dir tests/fixtures/phenotype_formatting

### Junction conversion from STAR output

`star2junc` converts each `SJ.out.tab` named in the manifest into the junction BED format LeafCutter2 reads, for cohorts whose BAMs are no longer available. Its output feeds `leafcutter2_cluster`.

**Timing**: ~1 s (on toy dataset)

In [ ]:
sos run pipeline/splicing_calling_leafcutter2.ipynb star2junc \
    --cwd output/leafcutter2 \
    --samples tests/fixtures/splicing_calling/leafcutter/protocol_example.sample_list.txt \
    --data-dir tests/fixtures/splicing_calling/leafcutter

## Command Interface

In [ ]:
sos run pipeline/splicing_calling_leafcutter2.ipynb -h

```
usage: sos run pipeline/splicing_calling_leafcutter2.ipynb
               [workflow_name | -t targets] [options] [workflow_options]
  workflow_name:        Single or combined workflows defined in this script
  targets:              One or more targets to generate
  options:              Single-hyphen sos parameters (see "sos run -h" for details)
  workflow_options:     Double-hyphen workflow-specific parameters

Workflows:
  leafcutter2
  leafcutter2_preprocessing
  leafcutter2_cluster
  star2junc

Global Workflow Options:
  --modular-script-dir code/script (as path)
  --cwd output (as path)
                        The output directory for generated files.
  --samples VAL (as path, required)
                        Sample meta data list
  --data-dir  path(f"{samples:d}")

                        Raw data directory, default to the same directory as
                        sample list
  --job-size 1 (as int)
                        For cluster jobs, number commands to run per job
  --walltime 5h
                        Wall clock time expected
  --mem 16G
                        Memory expected
  --numThreads 8 (as int)
                        Number of threads
  --container ''
                        Software container option

Sections
  leafcutter2_1, leafcutter2_preprocessing_1:
    Workflow Options:
      --anchor-len 8 (as int)
                        anchor length (default 8)
      --min-intron-len 50 (as int)
                        minimum intron length to be analyzed (default 50)
      --max-intron-len 500000 (as int)
                        maximum intron length to be analyzed (default 500000)
  leafcutter2_2:
    Workflow Options:
      --annotation-gtf VAL (as path, required)
                        GTF annotation with gene, transcript, CDS, start_codon
                        and stop_codon features
      --genome-fasta VAL (as path, required)
                        Genome assembly FASTA matching the GTF, with a .fai
                        index beside it
      --min-clu-reads 30 (as int)
                        minimum reads in a cluster (default 30 reads)
      --min-reads 5 (as int)
                        minimum reads for a junction to be considered for
                        clustering (default 5 reads)
      --min-read-std 0.5 (as float)
                        minimum standard deviation of reads across samples
                        (default 0.5)
      --min-clu-ratio 0.001 (as float)
                        minimum fraction of reads in a cluster that support a
                        junction (default 0.001)
      --max-intron-len 500000 (as int)
                        maximum intron length to be analyzed (default 500000,
                        the GTEx value, not LeafCutter2's 100000)
      --offset 1 (as int)
                        coordinate offset; 1 matches
                        leafcutter_cluster_regtools.R intron ends for regtools
                        .junc input
  leafcutter2_cluster:
    Workflow Options:
      --juncfiles VAL (as path, required)
                        text file listing one junction file path per line
      --annotation-gtf VAL (as path, required)
                        GTF annotation with gene, transcript, CDS, start_codon
                        and stop_codon features
      --genome-fasta VAL (as path, required)
                        Genome assembly FASTA matching the GTF, with a .fai
                        index beside it
      --min-clu-reads 30 (as int)
                        minimum reads in a cluster (default 30 reads)
      --min-reads 5 (as int)
                        minimum reads for a junction to be considered for
                        clustering (default 5 reads)
      --min-read-std 0.5 (as float)
                        minimum standard deviation of reads across samples
                        (default 0.5)
      --min-clu-ratio 0.001 (as float)
                        minimum fraction of reads in a cluster that support a
                        junction (default 0.001)
      --max-intron-len 500000 (as int)
                        maximum intron length to be analyzed (default 500000,
                        the GTEx value, not LeafCutter2's 100000)
      --offset 0 (as int)
                        coordinate offset; 0 for BED6 junction files, 1 for
                        regtools .junc
  leafcutter2_preprocessing_2:
  star2junc:
```

## Workflow implementation

In [ ]:
[global]
parameter: modular_script_dir = path('code/script')  # override with --modular-script-dir
# The output directory for generated files.
parameter: cwd = path("output")
# Sample meta data list
parameter: samples = path
# Raw data directory, default to the same directory as sample list
parameter: data_dir = path(f"{samples:d}")

# For cluster jobs, number commands to run per job
parameter: job_size = 1
# Wall clock time expected
parameter: walltime = "5h"
# Memory expected
parameter: mem = "16G"
# Number of threads
parameter: numThreads = 8
# Software container option
parameter: container = ""
from sos.utils import expand_size
cwd = path(f'{cwd:a}')

# Build the input file lists from the sample manifest (basic SoS input plumbing; no pandas so
# it runs in the base env, no def/handles left in globals so the lists pickle to task workers).
# regtools strandness codes: rf->RF, fr->FR, strand_missing->XS. Blank/"NA" drops that sample.
_lines = open(samples).read().splitlines()
_hdr = _lines[0].split('\t')
_rows = [dict(zip(_hdr, _ln.split('\t'))) for _ln in _lines[1:] if _ln.strip()]
_sm = {'rf': 'RF', 'fr': 'FR', 'strand_missing': 'XS'}
_blank = lambda v: (v or 'NA').strip() in ('', 'NA')
# A manifest may give either the long form (rf, fr, strand_missing) or the regtools code it maps
# to (RF, FR, XS). Anything else is rejected here rather than passed to regtools, which would
# take it as a literal and silently produce wrongly stranded junctions.
_ok = set(_sm) | set(_sm.values())
_bad = sorted({(_r.get('strand') or '').strip() for _r in _rows} - _ok)
if _bad:
    raise ValueError(f"unknown strand value(s) {_bad} in {samples}; expected one of {sorted(_ok)}")
# BAM rows and SJ rows are filtered independently, and every per-sample list is derived from
# its own filtered row set, so dropping a row with a blank or NA path cannot shift the
# strandness of the row after it.
_bam_rows  = [_r for _r in _rows if not _blank(_r.get('coord_bam_list'))]
_sj_rows   = [_r for _r in _rows if not _blank(_r.get('SJ_list'))]
sample_id  = [(_r.get('sample_id') or 'NA').strip() for _r in _bam_rows]
strandness = [_sm.get(_s, _s) for _s in [(_r.get('strand') or '').strip() for _r in _bam_rows]]
bam_data   = [f'{data_dir}/{_r["coord_bam_list"].strip()}' for _r in _bam_rows]
sj_data    = [f'{data_dir}/{_r["SJ_list"].strip()}' for _r in _sj_rows]
# Junction files are named after the BAM basename, so a collision would overwrite a sample.
_bn = [_b.rsplit('/', 1)[-1] for _b in bam_data]
if len(set(_bn)) != len(_bn):
    raise ValueError(f"duplicate BAM basenames in {samples}: junction outputs would collide")
del _lines, _hdr, _rows, _sm, _blank, _ok, _bad, _bam_rows, _sj_rows, _bn

### Junction extraction

Documentation: [LeafCutter2](https://github.com/leafcutter2/leafcutter2); the regtools parameter choices are [discussed here](https://github.com/davidaknowles/leafcutter/issues/127). This step is identical to `splicing_calling`'s first step, so junctions extracted by either module are interchangeable. `leafcutter2_1` extracts junctions from each BAM, `leafcutter2_preprocessing_2` writes the junction list instead of clustering.

In [ ]:
[leafcutter2_1, leafcutter2_preprocessing_1]
# anchor length (default 8)
parameter: anchor_len = 8
# minimum intron length to be analyzed (default 50)
parameter: min_intron_len = 50
# maximum intron length to be analyzed (default 500000)
parameter: max_intron_len = 500000
input: bam_data, group_by = 1, group_with = "strandness"
output: f'{cwd}/{_input:bn}.junc'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container=container
    bash ${modular_script_dir}/molecular_phenotypes/calling/regtools_junctions.sh \
        --bam ${_input} \
        --output ${_output} \
        --min-anchor ${anchor_len} \
        --min-intron ${min_intron_len} \
        --max-intron ${max_intron_len} \
        --strandness ${_strandness}

### Clustering, classification and quantification

`leafcutter2_2` clusters the extracted junctions and classifies each one as productive, unproductive, ambiguous or intergenic. `--keepleafcutter1` is always passed so that LeafCutter1-format tables are written alongside, which keeps the original `leafcutter_ds.R` usable on these clusters.

In [ ]:
[leafcutter2_2]
# GTF annotation with gene, transcript, CDS, start_codon and stop_codon features
parameter: annotation_gtf = path
# Genome assembly FASTA matching the GTF, with a .fai index beside it
parameter: genome_fasta = path
# prefix for every output file; defaults to the manifest name with all extensions stripped.
# Set it explicitly when two workflows write into one --cwd, or when the manifest name carries
# dots that are not extensions (cohort.v1.sample_list.txt would otherwise collapse to "cohort").
parameter: output_prefix = f'{samples:bnn}'
# minimum reads in a cluster (default 30 reads)
parameter: min_clu_reads = 30
# minimum reads for a junction to be considered for clustering (default 5 reads)
parameter: min_reads = 5
# minimum standard deviation of reads across samples (default 0.5)
parameter: min_read_std = 0.5
# minimum fraction of reads in a cluster that support a junction (default 0.001)
parameter: min_clu_ratio = 0.001
# maximum intron length to be analyzed (default 500000, the GTEx value, not LeafCutter2's 100000)
parameter: max_intron_len = 500000
# coordinate offset; 1 matches leafcutter_cluster_regtools.R intron ends for regtools .junc input
parameter: offset = 1
input: group_by = 'all'
output: counts = f'{cwd}/{output_prefix}.junction_counts.gz',
        ratios = f'{cwd}/{output_prefix}.cluster_ratios.gz',
        lc1_counts = f'{cwd}/leafcutter1_files/{output_prefix}_perind.counts.gz',
        lc1_numers = f'{cwd}/leafcutter1_files/{output_prefix}_perind_numers.counts.gz',
        classifications = f'{cwd}/clustering/{output_prefix}_junction_classifications.txt'
depends: annotation_gtf, genome_fasta, path(f'{genome_fasta}.fai')
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{cwd}/{output_prefix}.stderr', stdout = f'{cwd}/{output_prefix}.stdout', container=container
    set -e
    _juncfile=${cwd}/${output_prefix}.intron_usage_perind.junc
    rm -f $_juncfile
    for i in ${_input:r}; do
    echo $i >> $_juncfile ; done
    # Junction and annotation chromosome naming must agree or nothing classifies, and the
    # failure is silent: every junction simply comes back unannotated.
    jchr=$(zcat -f "$(head -n1 $_juncfile)" | head -n1 | cut -f1)
    achr=$(grep -v '^#' ${annotation_gtf} | head -n1 | cut -f1)
    case "$jchr" in chr*) jp=prefixed;; *) jp=bare;; esac
    case "$achr" in chr*) ap=prefixed;; *) ap=bare;; esac
    if [ "$jp" != "$ap" ]; then
        echo "ERROR: chromosome naming differs. Junctions use '$jchr', annotation uses '$achr'." >&2
        exit 1
    fi
    leafcutter2 \
        --juncfiles $_juncfile \
        --outprefix ${output_prefix} \
        --rundir ${cwd} \
        --annot ${annotation_gtf} \
        --genome ${genome_fasta} \
        --maxintronlen ${max_intron_len} \
        --minclureads ${min_clu_reads} \
        --minreads ${min_reads} \
        --minreadstd ${min_read_std} \
        --mincluratio ${min_clu_ratio} \
        --offset ${offset} \
        --keepleafcutter1
    # An empty result means the filters removed every cluster rather than that nothing was wrong.
    if [ "$(zcat ${_output['ratios']} | head -n2 | wc -l)" -lt 2 ]; then
        echo "ERROR: no clusters survived filtering. Lower --min-clu-reads (currently ${min_clu_reads})" >&2
        echo "       or --min-reads (currently ${min_reads}), or check that the junction files are not empty." >&2
        exit 1
    fi

### Clustering from an existing junction list

`leafcutter2_cluster` is an alternative entry point to the step above: it takes a junction file list directly rather than extracting junctions from BAMs. It writes the same outputs, so the two do not chain.

In [ ]:
[leafcutter2_cluster]
# text file listing one junction file path per line
parameter: juncfiles = path
# GTF annotation with gene, transcript, CDS, start_codon and stop_codon features
parameter: annotation_gtf = path
# Genome assembly FASTA matching the GTF, with a .fai index beside it
parameter: genome_fasta = path
# prefix for every output file; defaults to the manifest name with all extensions stripped.
# Set it explicitly when two workflows write into one --cwd, or when the manifest name carries
# dots that are not extensions (cohort.v1.sample_list.txt would otherwise collapse to "cohort").
parameter: output_prefix = f'{samples:bnn}'
# minimum reads in a cluster (default 30 reads)
parameter: min_clu_reads = 30
# minimum reads for a junction to be considered for clustering (default 5 reads)
parameter: min_reads = 5
# minimum standard deviation of reads across samples (default 0.5)
parameter: min_read_std = 0.5
# minimum fraction of reads in a cluster that support a junction (default 0.001)
parameter: min_clu_ratio = 0.001
# maximum intron length to be analyzed (default 500000, the GTEx value, not LeafCutter2's 100000)
parameter: max_intron_len = 500000
# coordinate offset. Required, with no default: it depends on the junction file format, and the
# wrong value corrupts the classification silently. Pass 0 for BED6, 1 for regtools .junc.
parameter: offset = int
# Junction files are declared as dependencies so a missing or changed one invalidates the run.
_junc_list = [_l.strip() for _l in open(juncfiles) if _l.strip()]
input: juncfiles
output: counts = f'{cwd}/{output_prefix}.junction_counts.gz',
        ratios = f'{cwd}/{output_prefix}.cluster_ratios.gz',
        lc1_counts = f'{cwd}/leafcutter1_files/{output_prefix}_perind.counts.gz',
        lc1_numers = f'{cwd}/leafcutter1_files/{output_prefix}_perind_numers.counts.gz',
        classifications = f'{cwd}/clustering/{output_prefix}_junction_classifications.txt'
depends: annotation_gtf, genome_fasta, path(f'{genome_fasta}.fai'), _junc_list
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{cwd}/{output_prefix}.stderr', stdout = f'{cwd}/{output_prefix}.stdout', container=container
    set -e
    _juncfile=${_input}
    # One offset applies to the whole run, so a list mixing BED6 and BED12 cannot be right for all.
    ncol=$(while read -r f; do [ -n "$f" ] && zcat -f "$f" | head -n1 | awk '{print NF}'; done < $_juncfile | sort -u | tr '\n' ' ')
    if [ "$(echo $ncol | wc -w)" -gt 1 ]; then
        echo "ERROR: junction files mix formats (column counts: $ncol). A single --offset cannot suit both." >&2
        exit 1
    fi
    # Junction and annotation chromosome naming must agree or nothing classifies, and the
    # failure is silent: every junction simply comes back unannotated.
    jchr=$(zcat -f "$(head -n1 $_juncfile)" | head -n1 | cut -f1)
    achr=$(grep -v '^#' ${annotation_gtf} | head -n1 | cut -f1)
    case "$jchr" in chr*) jp=prefixed;; *) jp=bare;; esac
    case "$achr" in chr*) ap=prefixed;; *) ap=bare;; esac
    if [ "$jp" != "$ap" ]; then
        echo "ERROR: chromosome naming differs. Junctions use '$jchr', annotation uses '$achr'." >&2
        exit 1
    fi
    leafcutter2 \
        --juncfiles $_juncfile \
        --outprefix ${output_prefix} \
        --rundir ${cwd} \
        --annot ${annotation_gtf} \
        --genome ${genome_fasta} \
        --maxintronlen ${max_intron_len} \
        --minclureads ${min_clu_reads} \
        --minreads ${min_reads} \
        --minreadstd ${min_read_std} \
        --mincluratio ${min_clu_ratio} \
        --offset ${offset} \
        --keepleafcutter1
    # An empty result means the filters removed every cluster rather than that nothing was wrong.
    if [ "$(zcat ${_output['ratios']} | head -n2 | wc -l)" -lt 2 ]; then
        echo "ERROR: no clusters survived filtering. Lower --min-clu-reads (currently ${min_clu_reads})" >&2
        echo "       or --min-reads (currently ${min_reads}), or check that the junction files are not empty." >&2
        exit 1
    fi

### Junction list only

`leafcutter2_preprocessing_2` writes the list of per-sample `.junc` files and stops, for cohorts whose clustering is done elsewhere.

In [ ]:
[leafcutter2_preprocessing_2]
# prefix for every output file; defaults to the manifest name with all extensions stripped.
# Set it explicitly when two workflows write into one --cwd, or when the manifest name carries
# dots that are not extensions (cohort.v1.sample_list.txt would otherwise collapse to "cohort").
parameter: output_prefix = f'{samples:bnn}'
input: group_by = 'all'
output: f'{cwd}/{output_prefix}.intron_usage_perind.junc'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container=container
    rm -f ${_output:r}
    for i in ${_input:r}; do
    echo $i >> ${_output:r} ; done

### Junction conversion from STAR output

`star2junc` converts each `SJ.out.tab` named in the manifest into the junction BED format LeafCutter2 reads. Use it when the BAMs are unavailable; note that `SJ.out.tab` records uniquely mapped junction reads only, so counts will not match those regtools extracts from the same library.

In [ ]:
[star2junc]
input: sj_data, group_by = 1
output: f'{cwd}/{_input:bn}.junc.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{_output:nn}.stderr', stdout = f'{_output:nn}.stdout', container=container
    leafcutter2-star2junc ${_input} ${_output}

### Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `ERROR: chromosome naming differs` | The junction files and the GTF use different chromosome naming (`22` against `chr22`). The preflight check stops the run before clustering. | Supply a GTF and FASTA whose naming matches the junction files. |
| `Could not find <chrom> (<strand>) in annotations`, followed by `ValueError: Cannot set a DataFrame with multiple columns to the single column SJClass` | The naming styles agree, so the preflight passes, but the GTF still does not cover some chromosome present in the junction files. Clustering completes and the crash comes later, in classification. | Supply a GTF and FASTA covering every chromosome in the junction files. |
| Productive fraction collapses; `UP` and `NE` dominate `junction_counts.gz` | `--offset` does not match the junction file format. | `1` for regtools `.junc`, `0` for BED6. See Input. |
| `ERROR: junction files mix formats` | The `--juncfiles` list names both BED6 and BED12 files, which need different offsets. | Convert them to one format, or cluster each set separately. |
| `the following arguments are required: --offset` | `leafcutter2_cluster` has no offset default by design. | Pass `--offset 0` for BED6 or `--offset 1` for regtools `.junc`. |
| `ERROR: no clusters survived filtering`, or `ValueError: Cannot set a DataFrame with multiple columns to the single column SJClass` with an empty `clustering/<prefix>_clusters` | No cluster met `--min-clu-reads`, so there was nothing to classify. Clustering reports success and the failure comes later. | Lower `--min-clu-reads`, or check the junction files actually carry reads in the region. |
| `exitcode=127` | `leafcutter2` is not on `PATH`. | Activate the pixi environment holding it before running, or supply `--container`. |
| `AttributeError: module 'leafcutter2.Transcript_tools' has no attribute 'ClassifySpliceJunction'`, after clustering reports success | A pre-2.0 LeafCutter2 is installed. Clustering works, classification does not. | Install LeafCutter2 2.0 or newer. See Input. |
| `leafcutter2-star2junc: command not found` | Same cause: that command only exists in 2.x. | Install LeafCutter2 2.0 or newer. |
| Introns above 100 kb missing | `--max-intron-len` left at LeafCutter2's default. | This module defaults to `500000`; check it has not been overridden. |
| Classification runs but every junction is `IN` | The GTF lacks `CDS`, `start_codon` or `stop_codon` features, or uses a tag this module does not auto-detect. | Use a GTF carrying those features; set `--transcript_type`, `--gene_name` or `--transcript_name` on the tool if the tags differ. |